In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/02_api_football_client

In [0]:
import json
from pathlib import Path
from typing import Any


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide league IDs, seasons, API settings, secrets, paths, and daily
# quota configuration are loaded from 00_project_config.
#
# HTTP communication, request pacing, rate-limit tracking, retries, error
# handling, and atomic JSON writes are provided by 02_api_football_client.

ENDPOINT = "players"

PAGES_ROOT = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/player_statistics/pages"
)

CONSOLIDATED_OUTPUT_FILE = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/player_statistics/raw_player_season_stats.json"
)

# Do not intentionally consume the entire daily API allowance.
#
# The API response headers remain the authoritative source once requests start,
# because some of today's quota may already have been used by other notebooks.
MAX_REQUESTS_PER_RUN = (
    API_FOOTBALL_DAILY_REQUEST_LIMIT
    - API_FOOTBALL_DAILY_REQUEST_RESERVE
)


# -----------------------------------------------------------------------------
# Local raw-file helpers
# -----------------------------------------------------------------------------

def get_page_file_path(
    league_id: int,
    season: int,
    page: int,
) -> Path:
    """
    Return the raw file path for one API-Football player page.

    The partition-like folder structure makes the raw extraction easy to
    inspect and allows ingestion to resume at page level across notebook runs.
    """

    return Path(
        PAGES_ROOT,
        f"league_id={league_id}",
        f"season={season}",
        f"page={page:04d}.json",
    )


def read_json_file(file_path: Path) -> dict[str, Any]:
    """Read and validate one locally persisted raw JSON object."""

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError(
            f"Expected JSON object in raw page file: {file_path}"
        )

    return payload


def validate_player_page(
    payload: dict[str, Any],
    *,
    league_id: int,
    season: int,
    expected_page: int,
) -> tuple[list[dict[str, Any]], int, int]:
    """
    Validate the minimum API-Football structure required by this ingestion.

    Returns:
        page_records
        current_page
        total_pages
    """

    page_records = payload.get("response")
    paging = payload.get("paging")

    if not isinstance(page_records, list):
        raise APIFootballError(
            "Expected API-Football 'response' to be a list for "
            f"league={league_id}, season={season}, page={expected_page}."
        )

    if not isinstance(paging, dict):
        raise APIFootballError(
            "Expected API-Football 'paging' to be an object for "
            f"league={league_id}, season={season}, page={expected_page}."
        )

    try:
        current_page = int(
            paging.get("current")
        )

        total_pages = int(
            paging.get("total")
        )

    except (TypeError, ValueError) as exc:
        raise APIFootballError(
            "Invalid API-Football pagination metadata for "
            f"league={league_id}, season={season}, page={expected_page}."
        ) from exc

    if current_page != expected_page:
        raise APIFootballError(
            "API-Football returned an unexpected page number. "
            f"Requested={expected_page}, returned={current_page}, "
            f"league={league_id}, season={season}."
        )

    if total_pages < 1:
        raise APIFootballError(
            "API-Football returned an invalid total page count "
            f"for league={league_id}, season={season}: {total_pages}"
        )

    if current_page > total_pages:
        raise APIFootballError(
            "API-Football returned current page greater than total pages "
            f"for league={league_id}, season={season}: "
            f"{current_page}/{total_pages}"
        )

    # For the selected top-five leagues/seasons we expect player data.
    #
    # Do not persist an unexpected empty response because that could make a
    # transient upstream problem look like a successfully completed page.
    if not page_records:
        raise APIFootballError(
            "API-Football returned zero players for "
            f"league={league_id}, season={season}, page={expected_page}."
        )

    return (
        page_records,
        current_page,
        total_pages,
    )


def get_saved_total_pages(
    league_id: int,
    season: int,
) -> int | None:
    """
    Return total pages from the saved first page, if available.

    Page 1 contains the pagination metadata required to determine whether a
    league-season extraction has already been completed.
    """

    first_page_file = get_page_file_path(
        league_id,
        season,
        1,
    )

    if not first_page_file.exists():
        return None

    payload = read_json_file(
        first_page_file
    )

    _, _, total_pages = validate_player_page(
        payload,
        league_id=league_id,
        season=season,
        expected_page=1,
    )

    return total_pages


def league_season_is_complete(
    league_id: int,
    season: int,
) -> bool:
    """
    Return True when every expected page for a league-season exists locally.
    """

    total_pages = get_saved_total_pages(
        league_id,
        season,
    )

    if total_pages is None:
        return False

    return all(
        get_page_file_path(
            league_id,
            season,
            page,
        ).exists()
        for page in range(
            1,
            total_pages + 1,
        )
    )


def all_extractions_complete() -> bool:
    """
    Return True when every configured league-season extraction is complete.
    """

    return all(
        league_season_is_complete(
            league_id,
            season,
        )
        for league_id in API_FOOTBALL_LEAGUES
        for season in API_FOOTBALL_SEASONS
    )


# -----------------------------------------------------------------------------
# Consolidation helper
# -----------------------------------------------------------------------------

def build_consolidated_player_file() -> int:
    """
    Build the existing consolidated raw-player file from persisted page files.

    No API requests are made here.

    Page-level API envelopes remain stored under PAGES_ROOT for source lineage.
    The consolidated file contains only player business records because that is
    the structure currently expected by the Bronze ingestion notebook.

    Returns:
        Number of player-statistic records written.
    """

    all_player_statistics: list[dict[str, Any]] = []

    for league_id in API_FOOTBALL_LEAGUES:
        for season in API_FOOTBALL_SEASONS:

            total_pages = get_saved_total_pages(
                league_id,
                season,
            )

            if total_pages is None:
                raise RuntimeError(
                    "Cannot build consolidated player file because "
                    f"league={league_id}, season={season} "
                    "has no saved first page."
                )

            for page in range(
                1,
                total_pages + 1,
            ):
                page_file = get_page_file_path(
                    league_id,
                    season,
                    page,
                )

                if not page_file.exists():
                    raise RuntimeError(
                        "Cannot build consolidated player file because "
                        f"page is missing: {page_file}"
                    )

                payload = read_json_file(
                    page_file
                )

                page_records, _, _ = validate_player_page(
                    payload,
                    league_id=league_id,
                    season=season,
                    expected_page=page,
                )

                all_player_statistics.extend(
                    page_records
                )

    if not all_player_statistics:
        raise RuntimeError(
            "Consolidated player-statistics dataset would be empty."
        )

    write_json_atomic(
        data=all_player_statistics,
        file_path=CONSOLIDATED_OUTPUT_FILE,
    )

    return len(
        all_player_statistics
    )


# -----------------------------------------------------------------------------
# Load API credential
# -----------------------------------------------------------------------------
# Credentials are retrieved from Databricks Secrets and never stored directly
# in notebook source code.

api_key = dbutils.secrets.get(
    scope=API_FOOTBALL_SECRET_SCOPE,
    key=API_FOOTBALL_SECRET_KEY,
)


# -----------------------------------------------------------------------------
# Ingestion state
# -----------------------------------------------------------------------------

requests_made_this_run = 0
pages_downloaded_this_run = 0
pages_skipped_existing = 0
completed_league_seasons = 0

stop_for_daily_budget = False


# -----------------------------------------------------------------------------
# Fetch player statistics
# -----------------------------------------------------------------------------
# Processing order is deterministic:
#
#   league -> season -> page
#
# Existing page files are treated as completed checkpoints and are never
# requested again.
#
# This makes the ingestion resumable across multiple days without maintaining
# a separate mutable checkpoint database.

with APIFootballClient(
    api_key=api_key,
    base_url=API_FOOTBALL_BASE_URL,
    min_request_interval_seconds=API_FOOTBALL_MIN_REQUEST_INTERVAL_SECONDS,
) as client:

    for league_id, league_name in API_FOOTBALL_LEAGUES.items():

        if stop_for_daily_budget:
            break

        for season in API_FOOTBALL_SEASONS:

            if stop_for_daily_budget:
                break

            print(
                "\n"
                "------------------------------------------------------------"
            )

            print(
                f"Processing {league_name} "
                f"(league_id={league_id}), season={season}"
            )

            # -----------------------------------------------------------------
            # Skip fully completed league-season combinations
            # -----------------------------------------------------------------

            if league_season_is_complete(
                league_id,
                season,
            ):
                completed_league_seasons += 1

                print(
                    "League-season already complete. "
                    "No API requests required."
                )

                continue


            # -----------------------------------------------------------------
            # Process pages
            # -----------------------------------------------------------------

            page = 1

            while True:

                page_file = get_page_file_path(
                    league_id,
                    season,
                    page,
                )

                # -------------------------------------------------------------
                # Reuse an already persisted page
                # -------------------------------------------------------------
                # Reading saved pagination metadata allows us to continue from
                # yesterday's stopping point without making another API call.

                if page_file.exists():

                    payload = read_json_file(
                        page_file
                    )

                    (
                        page_records,
                        current_page,
                        total_pages,
                    ) = validate_player_page(
                        payload,
                        league_id=league_id,
                        season=season,
                        expected_page=page,
                    )

                    pages_skipped_existing += 1

                    print(
                        f"Page {current_page}/{total_pages} already exists "
                        f"({len(page_records)} players). Skipping request."
                    )

                else:

                    # ---------------------------------------------------------
                    # Protect the configured per-run request budget
                    # ---------------------------------------------------------

                    if requests_made_this_run >= MAX_REQUESTS_PER_RUN:

                        print(
                            "\nConfigured daily request budget reached."
                        )

                        stop_for_daily_budget = True
                        break


                    # ---------------------------------------------------------
                    # Protect the actual API quota reported by API-Football
                    # ---------------------------------------------------------
                    # This becomes available after the first successful request
                    # made by this client.
                    #
                    # It matters because another ingestion notebook may already
                    # have consumed requests earlier in the same day.

                    if (
                        client.daily_remaining is not None
                        and client.daily_remaining
                        <= API_FOOTBALL_DAILY_REQUEST_RESERVE
                    ):
                        print(
                            "\nAPI-Football daily quota reached the "
                            "configured safety reserve."
                        )

                        stop_for_daily_budget = True
                        break


                    # ---------------------------------------------------------
                    # Fetch one page
                    # ---------------------------------------------------------

                    print(
                        f"Fetching page {page}..."
                    )

                    payload = client.get(
                        endpoint=ENDPOINT,
                        params={
                            "league": league_id,
                            "season": season,
                            "page": page,
                        },
                    )

                    requests_made_this_run += 1


                    # ---------------------------------------------------------
                    # Validate before persisting
                    # ---------------------------------------------------------

                    (
                        page_records,
                        current_page,
                        total_pages,
                    ) = validate_player_page(
                        payload,
                        league_id=league_id,
                        season=season,
                        expected_page=page,
                    )


                    # ---------------------------------------------------------
                    # Persist page atomically
                    # ---------------------------------------------------------
                    # A successful API page becomes a durable checkpoint before
                    # the notebook moves to the next request.

                    write_json_atomic(
                        data=payload,
                        file_path=str(page_file),
                    )

                    pages_downloaded_this_run += 1

                    print(
                        f"Saved page {current_page}/{total_pages}: "
                        f"{len(page_records)} players"
                    )

                    print(
                        "API daily requests remaining: "
                        f"{client.daily_remaining}"
                    )


                # -------------------------------------------------------------
                # End pagination for this league-season
                # -------------------------------------------------------------

                if current_page >= total_pages:
                    completed_league_seasons += 1

                    print(
                        f"Completed {league_name}, season={season}: "
                        f"{total_pages} pages."
                    )

                    break

                page = current_page + 1


# -----------------------------------------------------------------------------
# Run summary
# -----------------------------------------------------------------------------

total_combinations = (
    len(API_FOOTBALL_LEAGUES)
    * len(API_FOOTBALL_SEASONS)
)

print(
    "\n"
    "============================================================"
)

print("Player-statistics ingestion summary")

print(
    "============================================================"
)

print(
    f"Requests made this run: {requests_made_this_run}"
)

print(
    f"New pages downloaded: {pages_downloaded_this_run}"
)

print(
    f"Existing pages reused: {pages_skipped_existing}"
)

print(
    "Completed league-season combinations encountered this run: "
    f"{completed_league_seasons}/{total_combinations}"
)


# -----------------------------------------------------------------------------
# Consolidate only after the complete multi-day extraction finishes
# -----------------------------------------------------------------------------
# Until every league-season/page combination exists, the previous consolidated
# file is deliberately left untouched.
#
# This ensures Bronze never receives a partially refreshed dataset by mistake.

if all_extractions_complete():

    record_count = build_consolidated_player_file()

    print(
        "\nAll configured player-statistics extractions are complete."
    )

    print(
        f"Consolidated {record_count} player-statistic records."
    )

    print(
        f"Consolidated raw file: {CONSOLIDATED_OUTPUT_FILE}"
    )

else:

    print(
        "\nPlayer-statistics backfill is not complete yet."
    )

    print(
        "Run this notebook again after the API daily quota resets."
    )

    print(
        "Existing page files will be reused automatically; "
        "only missing pages will generate API requests."
    )